# `contextual-turn-encoder-base` **v2 (BERT-fiel)** — entrenamiento en M2

Gemela de [`02_train_contextual_m2.ipynb`](02_train_contextual_m2.ipynb) (v1), **cambiando solo la
arquitectura**: usa `build_model(arch="v2")` → `ContextualTurnModelV2`, el port fiel de BERT (post-LN,
sin residual). Todo lo demás (datos, held-out semilla 42, objetivo, hiperparámetros) es **idéntico** →
permite la comparación controlada v1 ↔ v2.

Entrena **AR** (causal) y **Bidi** (full-context), excluyendo los diálogos held-out del benchmark.

> Para que la comparación sea apples-to-apples, corré esto con el **mismo `CORPUS`** que usaste en la
> notebook 02 del v1. Diferencias v1↔BERT documentadas en [`docs/model/v2.md`](../../docs/model/v2.md).

## 1. Setup (rutas + imports)

In [ ]:
import os, sys, json, time
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader

# >>> ajustá estas rutas en tu M2 <<<
REPO = os.path.expanduser("~/Documents/GitHub/doctorado-unsl")
PKG = os.path.join(REPO, "packages/contextual-turn-embeddings")
RECIPE = os.path.join(PKG, "training/contextual-turn-encoder-base")
CONFIG_PATH = os.path.join(RECIPE, "config.yaml")
ANN = os.path.expanduser("~/Documents/GitHub/ANN-UNSL")

CORPUS = "1m"                                   # "full" | "1m"  (usá el mismo que en el v1)
if CORPUS == "full":
    META_PATH = os.path.join(REPO, "data/d2f-full/dialogs-full.pkl")
    BASE_EMB_PATH = os.path.join(REPO, "data/d2f-full/base_embeddings.npy")
else:                                           # 1m = colección del benchmark (dialogs-2.0)
    META_PATH = os.path.join(ANN, "data/dialogs-2.0.pkl")
    BASE_EMB_PATH = os.path.join(ANN, "data/embeddings_dialog2flow.npy")
# held-out reproducible (semilla 42) sobre la colección de 1M:
HELDOUT_META = os.path.join(ANN, "data/dialogs-2.0.pkl")
OUT = os.path.join(PKG, "models")               # gitignored

sys.path.insert(0, RECIPE)                       # heldout.py
import heldout as H
from contextual_turn_embeddings import (Config, DialogueDataset, EmbeddingRetrievalConfig,
    build_model, collate_dialogues, compute_objectives, get_device, resolve_losses_for_mode, set_seed)
from contextual_turn_embeddings.train import build_linear_warmup_scheduler
print("device:", get_device("mps"))

## 2. Cargar datos (memmap) y excluir held-out

In [ ]:
df = pd.read_pickle(META_PATH)[["dialogue_id", "turn_id", "speaker", "utterance"]].copy()
df["row_id"] = np.arange(len(df), dtype=np.int64)     # = índice en el memmap

emb = np.load(BASE_EMB_PATH, mmap_mode="r")           # NO entra en RAM
assert len(df) == len(emb), (len(df), len(emb))

df_1m = pd.read_pickle(HELDOUT_META)
heldout_ids = H.heldout_dialogue_ids(df_1m)
train_mask, _ = H.split_train_heldout(df, heldout_ids)
df_train = df[train_mask].reset_index(drop=True)      # row_id sigue apuntando al memmap

print(f"corpus {CORPUS}: {len(df)} turnos / {df.dialogue_id.nunique()} diálogos")
print(f"held-out: {len(heldout_ids)} diálogos | train: {len(df_train)} turnos / "
      f"{df_train.dialogue_id.nunique()} diálogos")

## 3. Función de entrenamiento (memmap, por modo) — usa `build_model(arch)`

In [ ]:
def train_variant(df_train, emb_memmap, base_cfg, arch, mode, num_layers, out_dir, retrieval=True):
    set_seed(base_cfg.training.seed)
    device = get_device(base_cfg.training.device)

    # val chica por diálogo (para la curva / mejor checkpoint)
    rng = np.random.default_rng(123)
    dids = pd.unique(df_train["dialogue_id"])
    val_dids = set(rng.choice(dids, size=max(1, int(len(dids) * 0.02)), replace=False))
    is_val = df_train["dialogue_id"].isin(val_dids).to_numpy()
    df_tr, df_va = df_train[~is_val], df_train[is_val]

    d = base_cfg.to_dict()
    d["model"]["arch"] = arch                          # <-- el ÚNICO cambio v1/v2
    cfg = Config.from_dict(d)
    cfg.model.attention_mode = mode
    cfg.model.num_layers = num_layers
    D = int(emb_memmap.shape[1])
    cfg.model.input_dim = cfg.model.hidden_dim = cfg.model.output_dim = D

    losses = resolve_losses_for_mode(cfg.losses, mode)
    if retrieval:
        losses.embedding_retrieval = EmbeddingRetrievalConfig(
            enabled=True, target=("masked" if mode == "bidirectional" else "next_turn"))

    mk = lambda dd: DialogueDataset(dd, emb_memmap, max_turns=cfg.data.max_turns,
                                    num_speakers=cfg.model.num_speakers, lazy=True)
    tr_loader = DataLoader(mk(df_tr), batch_size=cfg.training.batch_size, shuffle=True,
                           num_workers=0, collate_fn=collate_dialogues)
    va_loader = DataLoader(mk(df_va), batch_size=cfg.training.batch_size, shuffle=False,
                           num_workers=0, collate_fn=collate_dialogues)

    model = build_model(cfg.model).to(device)          # arch="v2" -> ContextualTurnModelV2
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.training.learning_rate,
                            weight_decay=cfg.training.weight_decay)
    total = max(1, len(tr_loader) * cfg.training.epochs)
    sched = build_linear_warmup_scheduler(opt, int(total * cfg.training.warmup_ratio), total)
    print(f"[{arch}/{mode}] {type(model).__name__} | "
          f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M params", flush=True)

    def move(b):
        o = dict(b)
        o["embeddings"] = b["embeddings"].to(device)
        o["attention_mask"] = b["attention_mask"].to(device)
        if b.get("speaker_ids") is not None:
            o["speaker_ids"] = b["speaker_ids"].to(device)
        return o

    @torch.no_grad()
    def validate():
        model.eval(); set_seed(999); tot = n = 0
        for b in va_loader:
            b = move(b); out = compute_objectives(model, b, losses)
            bs = b["embeddings"].shape[0]; tot += float(out["total"].detach().cpu()) * bs; n += bs
        model.train(); return tot / max(1, n)

    os.makedirs(out_dir, exist_ok=True)
    logf = open(os.path.join(out_dir, "trainlog.jsonl"), "w"); best = float("inf"); t0 = time.time()
    for ep in range(1, cfg.training.epochs + 1):
        model.train(); run = 0.0; te = time.time()
        for i, b in enumerate(tr_loader):
            b = move(b); opt.zero_grad(set_to_none=True)
            loss = compute_objectives(model, b, losses)["total"]; loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.training.gradient_clip_norm)
            opt.step(); sched.step(); run += float(loss.detach().cpu())
            if i % cfg.training.log_interval == 0:
                print(f"[{arch}/{mode}] ep{ep} {i}/{len(tr_loader)} loss={float(loss.detach().cpu()):.4f}", flush=True)
        rec = {"arch": arch, "mode": mode, "epoch": ep,
               "train_loss": round(run / max(1, len(tr_loader)), 5),
               "val_loss": round(validate(), 5), "epoch_sec": round(time.time() - te, 1)}
        print("EPOCH", json.dumps(rec), flush=True); logf.write(json.dumps(rec) + "\n"); logf.flush()
        model.save_pretrained(out_dir, training_args=rec)
        if rec["val_loss"] < best:
            best = rec["val_loss"]; model.save_pretrained(os.path.join(out_dir, "best"), training_args=rec)
    cfg.to_yaml(os.path.join(out_dir, "config.yaml"))
    print(f"[{arch}/{mode}] DONE best_val={best:.5f} min={(time.time()-t0)/60:.1f}", flush=True)
    return model

## 4. Entrenar AR + Bidi (v2)
Mismo recipe que el v1 (config.yaml: `epochs=10`, batch 128, contrastivo co-primario).

In [ ]:
base_cfg = Config.from_yaml(CONFIG_PATH)
ARCH = "v2"
NAME = "contextual-turn-encoder-base"
NLAYERS = 6 if CORPUS == "full" else 4         # igual que el v1

# (El residual de config.yaml lo ignora el v2; el v2 es post-LN sin residual.)
m_ar = train_variant(df_train, emb, base_cfg, ARCH, "autoregressive", NLAYERS,
                     os.path.join(OUT, f"{NAME}-{ARCH}-ar-{CORPUS}"))
m_bidi = train_variant(df_train, emb, base_cfg, ARCH, "bidirectional", NLAYERS,
                       os.path.join(OUT, f"{NAME}-{ARCH}-bidi-{CORPUS}"))
print("Checkpoints en:", OUT)

## 5. Después

- **Curvas / comparación v1↔v2:** `python training/contextual-turn-encoder-base/plot_v1_vs_v2.py`
  → `conversational-ann/results/v1_vs_v2_curves.png`.
- **Act-match v1 vs v2:** `python packages/conversational-ann/scripts/eval_prelim.py --corpus 1m
  --phase encode` y luego `--phase metric` (ya maneja v1 y v2).
- Resumen: `conversational-ann/results/v1_vs_v2_results.md`.
- Pesos: no van a git (`models/` gitignored) → asset de un GitHub Release, como el v1.